# 21 — GloVe
**Goal:** Understand global word co-occurrence embeddings and load pre-trained vectors.

GloVe ("Global Vectors") is a **count-based** embedding family: it builds a global word–word co-occurrence matrix over a huge corpus, then factorizes it so each word gets a dense vector whose geometry mirrors co-occurrence statistics. Unlike the predictive Word2Vec of Ch. 19, GloVe's signal comes from *global* counts rather than sliding-window context, which tends to give cleaner behavior on similarity tasks.

**Why it matters for resumes / ATS:** pre-trained GloVe vectors are an instant, dependency-free skill-similarity layer — no training data, no GPU. A matcher can compare `tensorflow` to related words or normalize synonyms by cosine distance instead of exact string match. It is the cheap, fast baseline that the rest of Block D (Ch. 22–24) builds on and then beats.

## 1. How GloVe Works

The key insight: the **ratio** of co-occurrence probabilities carries meaning, not the raw counts. If `P(k | ice)` is much larger than `P(k | steam)`, then word `k` behaves like "solid" (ice-like); the reverse picks out "gas"; when the two probabilities are nearly equal (`water`), the word is uninformative for distinguishing the pair. GloVe learns vectors whose dot products reproduce these log-probability ratios across the whole vocabulary.

**What the code does:** this cell is a pure-print cheat-sheet, not a model. It restates the core idea, walks through the ice/steam example as a preview of the math, and ends by hammering the takeaway: the co-occurrence *probability ratio* — not the raw count — is what encodes meaning.

**Try it:** the `solid` / `gas` / `water` trio is the classic illustration from the GloVe paper. Once §2 loads the vectors, probe it directly with `glove.most_similar(positive=["ice"], negative=["steam"])` and see which of the three ranks highest.

In [ ]:
print('''GloVe = Global Vectors for Word Representation
Key idea: word vectors learned from global co-occurrence counts across the corpus.
Unlike Word2Vec (predictive, local context), GloVe is count-based (global matrix factorization).

  P(k | ice) >> P(k | steam)  ->  k = "solid"
  P(k | steam) >> P(k | ice)  ->  k = "gas"
  P(k | ice) ≈ P(k | steam)   ->  k = "water" (uninformative)

Ratio of co-occurrence probabilities encodes meaning.'')

## 2. Loading Pre-trained GloVe

`gensim.downloader` fetches ready-made vectors over HTTP and caches them locally, so "loading GloVe" is a one-liner. The notebook uses `glove-twitter-25`: 25 dimensions, trained on ~2B tweets — small enough to download fast (~105 MB) and probe interactively.

**What the code does:** `api.load("glove-twitter-25")` downloads on first use, then:
- prints the vocabulary size — `len(glove)` gives **1,193,514** tokens on the reference run;
- prints `glove['python'].shape` — a 25-dimensional vector `(25,)`;
- prints the first 10 dimensions of `python`'s vector — raw floats such as `-0.256 -0.223 0.026 0.229 ...` (Twitter vectors are not normalized to a pretty range).

If the download fails, the `except ValueError` branch lists every model `api.info()` knows about so you can pick a different one.

**Try it:** swap the model string for `glove-wiki-gigaword-100` — 100 dims trained on Wikipedia, noticeably better on technical/professional language, at the cost of a larger download.

In [ ]:
import gensim.downloader as api
# Download the small 50d GloVe model (fast download, ~50MB)
print("Downloading GloVe (glove-twitter-25)...")
try:
    glove = api.load("glove-twitter-25")
    print(f"Vocabulary size: {len(glove)}")
    print(f"Vector dims: {glove['python'].shape}")
    print(f"First 10 dims of 'python': {glove['python'][:10].round(3)}")
except ValueError as e:
    print(f"""Download issue: {e}
But we can demo with available vectors. Let's check what's available:""")
    print(api.info()['models'].keys())

## 3. GloVe Word Similarity

`most_similar(w)` returns the vocabulary's highest **cosine similarity** neighbors of `w` — the words geometrically closest to it. That single call is the whole "semantic search" primitive: no synonym list, no regex, just vector distance.

**What the code does:** loops `["python", "java", "data", "science", "engineer"]` and prints the top-3 neighbors of each. The reference run on `glove-twitter-25` printed:

| Query | Top neighbors (score) |
|---|---|
| `python` | matrix (0.873), electronic (0.859), osx (0.858) |
| `java` | drupal (0.886), linux (0.867), electronic (0.858) |
| `data` | mobile (0.898), software (0.867), search (0.863) |
| `science` | english (0.920), research (0.918), psychology (0.911) |
| `engineer` | specialist (0.958), developer (0.955), administrator (0.943) |

Two honest observations: `engineer → specialist/developer` is exactly the resume-relevant signal we want, but the Twitter corpus is noisy (`science → english`, `python → matrix`). For professional text, §2's `glove-wiki-gigaword-100` behaves better. The `except NameError` guard only fires if §2 never ran in this session.

In [ ]:
# Similarity queries
try:
    for w in ["python", "java", "data", "science", "engineer"]:
        similar = glove.most_similar(w, topn=3)
        print(f"Similar to '{w}':")
        for word, score in similar:
            print(f"  {word:12s} {score:.3f}")
except NameError:
    print("GloVe not loaded. Install with: pip install gensim")
    print("then: import gensim.downloader; glove = api.load('glove-twitter-25')")

## 4. GloVe Analogies

Word-vector arithmetic encodes relations: if `king − man + woman ≈ queen`, then adding and subtracting vectors moves you along a semantic axis. `most_similar(positive=[...], negative=[...])` implements exactly that — positives are added, negatives subtracted, and the closest vocabulary words are returned.

**What the code does:** computes `python + developer − language` and prints the top-3. On the reference run with `glove-twitter-25` the result was **vmware (0.896), cnc (0.876), hyperion (0.854)** — not the textbook "programmer" answer. That is the lesson: analogies are *corpus-dependent*, and a 25-d Twitter model is too small and too colloquial to reproduce clean relations. The `except NameError` branch prints the intended concept (`python − language + developer → programmer`) for sessions where §2 never loaded the vectors.

**Try it:** the classic `berlin − germany + france ≈ paris` relation fails on many small models too — run it and watch the answer change with the corpus.

In [ ]:
try:
    result = glove.most_similar(positive=["python", "developer"], negative=["language"], topn=3)
    print(f"python + developer - language =")
    for w, s in result: print(f"  {w:12s} {s:.3f}")
except NameError:
    print("GloVe not loaded in this session. Concept demo:")
    print("  python - language + developer → programmer (analogy)")

## 5. GloVe vs Word2Vec — When to Use Which

Both families produce **static** (context-free) word vectors; they differ in *how* they learn them. Word2Vec is **predictive** — a shallow network learns to guess a word from its neighbors, so it trains fast and works on small corpora. GloVe is **count-based** — it factorizes the global co-occurrence matrix, so it needs the full corpus statistics but often yields smoother similarity behavior.

**What the code does:** prints a side-by-side comparison table (type, training signal, speed, memory, pre-trained availability). The practical translation for this project:

| Situation | Choose |
|---|---|
| Tiny corpus, fast iteration, local training | Word2Vec (Ch. 19) |
| Pre-trained quality, similarity tasks | GloVe (this chapter) |
| Phrases, OOV words, context sensitivity | Sentence Transformers (Ch. 22) |

For resume analysis the printed verdict is right: GloVe's global statistics usually beat Word2Vec on *skill similarity* — but both lose to contextual models once phrases and abbreviations enter the picture.

In [ ]:
print('''
╔══════════════════════════════════════════════════════════╗
║   GloVe vs Word2Vec                                      ║
╠══════════════════════════════════════════════════════════╣
║                    Word2Vec         GloVe                ║
║──────────────────────────────────────────────────────────║
║ Type              Predictive         Count-based         ║
║ Training          Local context      Global co-occurrence║
║ Speed             Faster (small)     Slower (needs full  ║
║                                      co-occurrence mat)  ║
║ Performance       Good on analogy    Better on similarity║
║ Memory            Small              Large (full matrix) ║
║ Pre-trained       Many options       Many options        ║
╚══════════════════════════════════════════════════════════╝

For resume analysis: GloVe often better for skill similarity tasks.''')

## Key Insight: GloVe captures global word statistics. Use pre-trained for resume similarity tasks.

**Static vectors are the cheapest semantic layer you can bolt onto a matcher — load them, don't train them.**

GloVe turns co-occurrence statistics into geometry: `engineer` sits near `specialist` and `developer`, and a cosine threshold becomes an "is this skill close enough" test with zero training data. The cost is that vectors are frozen per word — no phrase handling, no context, no OOV coverage. That is exactly what Ch. 22's Sentence Transformers fix, and Ch. 23 benchmarks both families head-to-head on resume-specific pairs.